# Chapter 6: Fine-tuning a GPT for classification

In the last chapter we gave our GPT a way to learn, a way to measure that learning, and a way to load OpenAI's public GPT-2 weights. At that point the model was a **general language model**: at every position it scored 50,257 possible next tokens. That is a powerful skill, but it is not the same as answering a yes/no question.

This chapter is about **fine-tuning** — taking a model that already understands language and specialising it for one narrower job. The job here is **classification**: read a whole piece of text, then pick a label. A classic example is spam detection: the two labels are *spam* and *not spam*.

Pretraining and fine-tuning are easy to mix up, so here is the difference in one glance:

| | Pretraining (Chapter 4) | Fine-tuning (this chapter) |
|---|---|---|
| **Question the model answers** | "What token comes next?" | "Which label is this whole text?" |
| **Output size** | `vocab_size` (50,257 scores at every position) | `num_classes` (2 scores for the whole sequence) |
| **Target** | a token ID at every position | a single class ID per example |
| **Data** | huge piles of raw text | a smaller labelled spreadsheet |
| **What we update** | (in principle) every weight | often just a tiny new head, with the backbone frozen |

Why not throw the pretrained model away and train a classifier from scratch? Because a labelled spam dataset might have a few thousand rows, while GPT-2 was pretrained on a huge slice of the internet. The transformer already knows what English looks like. Fine-tuning reuses that knowledge and only teaches the last step: how to turn a meaning into a label.

Here is the path we will follow:

- **A classification GPT**: keep the pretrained embeddings and transformer blocks, replace the 50,257-way next-token head with a small linear layer
- **Last-token pooling**: turn a whole sequence into one vector by reading the final position
- **The pretrained backbone**: bring `real_gpt` in from Chapter 4
- **Freezing**: lock the 124-million-parameter body so only the new head can learn
- **A labelled dataset**: encode texts, pad them to one shared length, pair each row with a class ID
- **Accuracy**: a second score alongside loss, because "how many labels did we get right?" is what this task is about
- **The fine-tuning loop**: the same training steps as Chapter 4, but with class logits instead of next-token logits

A quick reminder about the new shapes, because they change in this chapter:

- **input token IDs** still have shape `(batch, tokens)`
- **hidden states** after the transformer still have shape `(batch, tokens, emb_dim)`
- **classifier logits** have shape `(batch, num_classes)` — one pair of scores per sequence, not per token
- **labels** have shape `(batch,)` — one class ID per sequence

## Step 1: Modifying the GPT architecture for classification

The pretrained `GPTMODEL` ends with `out_head`, a linear layer that maps each token's `emb_dim` vector to `vocab_size` numbers. That head is perfect for writing, and useless for "spam or not spam". We do not need 50,257 options. We need **2**.

`GPTForClassification` wraps a finished GPT and keeps the parts that understand language:

- `tok_emb` and `pos_emb` — the same token and position embeddings
- the transformer stack — the pretrained blocks that mix information across tokens
- `drop_emb` and `final_ln` — the same regularisation and final normalisation

Then it throws away the next-token head and builds a new one:

```
classifier_head = Linear(emb_dim, num_classes)
```

`base_model.out_head.in_features` is how we read `emb_dim` out of the pretrained model (768 for GPT-2 124M) without hard-coding it. `num_classes=2` makes this a binary classifier. The same class would work for 3, 10, or 100 labels; only this number would change.

### What happens in `forward`

The first half is the GPT you already know:

1. `in_idx` has shape `(batch, seq_len)`.
2. Token embeddings and position embeddings are added, then dropped out, giving shape `(batch, seq_len, emb_dim)`.
3. The pretrained transformer blocks and the final LayerNorm refine those vectors. Every token still has its own `emb_dim` features.

The second half is the new idea:

4. `last_token_vectors = x[:, -1, :]` keeps **only the last position**. Shape goes from `(batch, seq_len, emb_dim)` to `(batch, emb_dim)`.
5. `classifier_head` maps that vector to `num_classes` raw scores. Shape is now `(batch, num_classes)`.

Those scores are still **logits**, the same word as in Chapter 4. Softmax would turn them into probabilities that the text is class 0 or class 1. Cross-entropy will do that internally during training.

### Why the last token?

A GPT is **causal**: token 5 may look at tokens 0 to 5, but never at token 6. By the time the model reaches the final position, that one vector has been allowed to attend to **every token in the message**. It is the model's running summary of the whole input.

BERT-style models add a special `[CLS]` token at the start and classify from there. A GPT already has a natural summary position: the last token. That is why this architecture is sometimes called *last-token classification*.

If the message was padded on the right with end-of-text tokens (we will do that in Step 4), the last position may itself be a pad token. That is still useful. Because attention is causal, that pad token has looked at every real token before it, so its vector still carries a view of the whole message.

In [6]:
import torch
import torch.nn as nn 

class GPTForClassification(nn.Module):
    def __init__(self, base_model, num_classes=2):
        super().__init__()
        self.tranformers= base_model.trf_blocks
        self.tok_emb= base_model.tok_emb
        self.pos_emb= base_model.pos_emb
        self.drop_emb= base_model.drop_emb
        self.final_ln= base_model.final_ln 

        self.classifier_head= nn.Linear(base_model.out_head.in_features, num_classes)

    def forward(self, in_idx):
        b,seq_len= in_idx.shape

        tok_embeds= self.tok_emb(in_idx)
        pos_embeds=self.pos_emb(torch.arrange(seq_len, device=in_idx.device))
        x= self.drop_emb(tok_embeds+pos_embeds)

        x=self.trf_blocks(x) if hasattr(self, 'trf_blocks') else self.transformer(x)
        x= self.final_ln(x)

        last_token_vectors = x[:, -1, :]

        logits= self.classifier_head(last_token_vectors)
        return logits 



## Step 2: Bringing in the pretrained GPT from Chapter 4

`GPTForClassification` is only a wrapper. It needs a finished `base_model` whose embeddings and transformer blocks already know English. That object is `real_gpt` from Chapter 4: our own `GPTMODEL` class, filled with the official GPT-2 124M weights.

`%run` is the same Jupyter magic we used to pull Chapter 3 into Chapter 4. It executes `../04_pretraining_and_loss_evaluation/llm_training.ipynb` from top to bottom, as if every cell of that notebook had been pasted here. When it finishes, the names we need are sitting in memory:

- `GPTMODEL` and `GPT_CONFIG_REAL` — the architecture and the 124M config
- `real_gpt` — that architecture with OpenAI's weights already mapped in
- `assign_weights`, `calc_loss_loader`, `generate_text_simple` — helpers we will reuse

Because the whole Chapter 4 notebook runs, and Chapter 4 itself runs Chapter 3, you will see a long stream of older printouts: toy losses, LayerNorm checks, Hugging Face download progress, the inference shape `(1, 3, 50257)`. That is expected. Scroll past them. What we care about is `real_gpt` being ready for the next cell to wrap.

One important detail: Chapter 4 left `real_gpt` in `eval` mode after the inference test. We will switch training back on later, but only for the tiny head we are about to add.

In [3]:
%run ../04_pretraining_and_loss_evaluation/llm_training.ipynb

logits shape: torch.Size([2, 3, 5])
targets shape: torch.Size([2, 3])
flattend logits shape: torch.Size([6, 5])
flattened targets shape: torch.Size([6])
calculated training loss: 0.45796191692352295
input tensor shape: torch.Size([2, 3, 5])
output tensor shape: torch.Size([2, 3, 5])

 normalised output mean (should be close to 0):
 tensor([[[ 2.3842e-08],
         [ 0.0000e+00],
         [ 1.1921e-08]],

        [[ 4.7684e-08],
         [ 3.5763e-08],
         [-4.7684e-08]]], grad_fn=<MeanBackward1>)

 normalised output var(should be close to 1):
 tensor([[[1.0000],
         [1.0000],
         [0.9999]],

        [[1.0000],
         [0.9999],
         [1.0000]]], grad_fn=<VarBackward0>)
input batch shape: torch.Size([8, 4, 256])
ffn output shape: torch.Size([8, 4, 256])
Note: you may need to restart the kernel to use updated packages.
inputs shape: torch.Size([3, 3])
query: tensor([0.4400, 0.1500, 0.8900])
raw attention scores: tensor([1.0082, 0.9599, 0.4495])
attention weights(percen

/home/softsensor/miniconda3/envs/llm_from_scratch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 148/148 [00:00<00:00, 1664.16it/s]


openai GPT2 layers found: 148

--- Sample Keys Mismatch Comparision ---
openai layer name: wte.weight
structural status: openai pretrained weights successfully mapped and injected
inferende pass result: logits shape is torch.Size([1, 3, 50257])


## Step 3: Freezing the backbone

We now wrap `real_gpt` in `GPTForClassification`. At this moment **every** copied layer is still trainable, including the 12 transformer blocks. Updating all 124 million parameters on a small spam CSV would almost certainly **overfit**: the model would memorise the training rows and fail on new messages. It would also be slow.

The standard first move is to treat the pretrained GPT as a **frozen feature extractor** and train only the new head. This is often called a **linear probe** (or last-layer fine-tuning):

1. The frozen body turns each message into a 768-dimensional meaning vector (the last-token hidden state).
2. The tiny trainable head learns a straight-line decision on top of those vectors: spam or not spam.

The cell below does that in three lines:

1. **`requires_grad = False` on every parameter.** This locks the embeddings, every transformer block, the final LayerNorm, and the new head.
2. **`requires_grad = True` on `classifier_head` only.** We unlock the one layer that did not come from GPT-2.
3. **Count `requires_grad` parameters.** You should see **1,538**. That number is not a mystery:

$$
768 \times 2 + 2 = 1{,}538
$$

`classifier_head` is `Linear(768, 2)`: 768 × 2 weights, plus 2 bias terms. Everything else is frozen, so it does not count.

A useful way to picture it:

```
[ frozen GPT-2 body: ~124,000,000 params ]
              |
              v
     last-token vector (768,)
              |
              v
[ trainable classifier_head: 1,538 params ]
              |
              v
     logits (2,)   spam / not spam
```

Later, if a linear probe is not accurate enough, you can unfreeze the last transformer block or two and fine-tune a little more of the body. This notebook starts with the safest, cheapest version: **body locked, head unlocked**.

In [7]:
model_classifier= GPTForClassification(real_gpt, num_classes=2)

for param in model_classifier.parameters():
    param.requires_grad=False
for param in model_classifier.classifier_head.parameters():
    param.requires_grad=True

trainable_params= sum(p.numel() for p in model_classifier.parameters() if p.requires_grad)
print("total trainable parameters for fine tuning:", trainable_params)

total trainable parameters for fine tuning: 1538


## Step 4: Preparing a classification dataset

A language-model dataset (Chapter 1) is a stream of tokens with a **next-token target at every position**. A classification dataset is a list of whole messages, each with **one label**.

`SpamDataset` turns a CSV file into that list. It expects two columns:

- `text` — the raw message, as a string
- `label` — the class ID, usually `0` or `1` for a binary problem

PyTorch talks to data through a small contract. Any `Dataset` must implement:

- `__len__` — how many examples there are
- `__getitem__(idx)` — return example number `idx` as tensors the model can eat

### From strings to padded token IDs

1. **`pd.read_csv`** loads the spreadsheet into a table.
2. **`tokenizer.encode(text)`** turns every message into a list of GPT-2 token IDs. Different messages produce different lengths. That is a problem, because a batch has to be a single rectangular tensor.
3. **`max_length`** is the shared length we pad everyone to. If you pass `None`, the dataset uses the longest message in the file. If you pass a number (for example, 128), every row is capped or padded to that width. It must stay within the model's `context_length` (1,024 for the real GPT-2 we loaded).
4. **Right-padding with `tokenizer.eot_token`**. GPT-2's end-of-text token (`<|endoftext|>`) is appended until the row is `max_length` long. Using a real special token, rather than `0`, gives the model an honest signal: "the message ended here". Because we pad on the right, the **last** position of a short message is this end-of-text token — and that is exactly the position `GPTForClassification` reads. Causal attention means that token has already looked at every real token before it, so it is a natural place to summarise the message.
5. The lists become tensors: `input_ids` of shape `(num_rows, max_length)` and `labels` of shape `(num_rows,)`.

`__getitem__` then returns one pair, `(input_ids[idx], labels[idx])`. A `DataLoader` can wrap this dataset, shuffle it, and serve batches of shape `(batch, max_length)` plus labels of shape `(batch,)`.

That label tensor is the whole point of the chapter. In pretraining the target was another sequence of token IDs. Here the target is a single integer: the class.

In [10]:
import torch
from torch.utils.data import Dataset 

class SpamDataset(Dataset):
    def __init__(self, csv_file, tokenizer, max_length=None):
        import pandas as pd 
        self.data=pd.read_csv(csv_file)
        self.encoded_texts= [tokenizer.encode(text) for text in self.data["text"]]

        if max_length is None:
            self.max_length=max(len(t) for t in self.encoded_texts)
        else:
            self.max_length= max_length 
        for i,text in enumerate(self.encoded_texts):
            if len(text)< self.max_length:
                padding_length= self.max_length-len(text)
                self.encoded_texts[i]+=[tokenizer.eot_token]* padding_length

        self.input_ids= torch.tesnsor(self.encoded_texts)
        self.labels= torch.tensor(self.data["label"].values)
    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.labels[idx]


## Step 5: Measuring accuracy

Loss tells us how confidently wrong the model is. For classification we also want a number a human can read without thinking: **out of 100 messages, how many labels did we get right?** That number is **accuracy**.

$$
\text{accuracy} = \frac{\text{correct predictions}}{\text{total examples}}
$$

A model can have a decent loss and still flip the wrong class on a lot of examples, or a slightly worse loss and a better accuracy. Tracking both keeps us honest. Loss is what we train on. Accuracy is what we report.

`calc_accuracy_loader` is the classification cousin of `calc_loss_loader` from Chapter 4. The safety rules are the same:

- **`model.eval()`** turns dropout off so a given message always gets the same prediction.
- **`torch.no_grad()`** skips the computation graph. We are scoring, not learning.
- **`num_batches`** lets us score a fast snapshot instead of the whole loader during a long run.

The new part is how a prediction is made:

1. `logits` have shape `(batch, num_classes)`. For spam vs not-spam that is `(batch, 2)`.
2. `torch.argmax(logits, dim=-1)` looks at the last dimension and picks the class with the larger score. The result has shape `(batch,)`: one predicted ID per message.
3. Comparing those IDs with `targets` gives a boolean tensor. Summing it counts how many messages were labelled correctly. `total_examples` grows by the batch size so we know the denominator.

A random binary classifier sits at **50%**. A useful spam filter needs to climb well above that on the **validation** loader, not only on the training loader. Training accuracy alone can lie: the head might memorise the training rows. Validation accuracy is the number we trust.

In [2]:
import torch 
def calc_accuracy_loader(dataloader, model, device, num_batches=None):
    model.eval()
    correct_predictions, total_examples=0,0
    for i, (inputs, targets) in enumerate(dataloader):
        if num_batches and i>=num_batches:
            break
        inputs,targets=input.to(device), targets.to(device)
        with torch.no_grad():
            logits=model(inputs)
        prediction=torch.argmax(logits, dim=-1)

        correct_predictions+=(predictions==targets).sum().item()
        total_examples+= targets.numel()

    return correct_predictions

## Step 6: The classification training loop

The outer loop should look familiar from Chapter 4. We still walk epochs, still call `zero_grad` / forward / loss / `backward` / `step`, and still peek at the validation set every `eval_freq` epochs. Three things change, and they are the whole point of this chapter.

### 1. The logits are per sequence, not per token

`model(inputs)` now returns shape `(batch, num_classes)`, not `(batch, tokens, vocab_size)`. The labels are shape `(batch,)`. Cross-entropy can consume those two tensors directly:

```
loss = cross_entropy(logits, targets)
```

There is no `flatten(0, 1)`. Flattening was the trick that turned every token position into its own classification problem. Here the sequence is already one example.

The formula is the same one from Chapter 4, just with a shorter score vector $z$ (2 numbers instead of 50,257) and one $z$ per message:

$$\mathcal{L} = -\log \, \text{softmax}(z)_{y}$$

### 2. We track accuracy as well as loss

After each evaluation tick the loop records four numbers:

| List | What it is |
|---|---|
| `train_losses` / `val_losses` | average cross-entropy, via `calc_loss_loader` from Chapter 4 |
| `train_accs` / `val_accs` | how many labels were right, via `calc_accuracy_loader` |

A healthy run has **both** training and validation accuracy climbing, and **both** losses falling. If training accuracy races to 100% while validation accuracy stalls, the head is memorising the training CSV.

`eval_iter` is the same speed knob as before: score only this many batches when printing a progress line.

### 3. Almost none of the model is actually moving

We still pass the full `model_classifier` into the optimizer. Because we froze the backbone, AdamW only receives gradients for `classifier_head`. The embeddings, the 12 transformer blocks, and the final LayerNorm stay exactly as GPT-2 left them. That is how 1,538 trainable numbers can specialise a 124-million-parameter model.

The function returns the four lists so you can plot train vs. validation loss and accuracy against the epoch number. Those two pictures together are how you decide whether fine-tuning worked.

In [3]:
import torch 
def train_classifier_model(model, train_loader, val_loader, optimizer, device, num_epochs, eval_freq, eval_iter):
    train_losses, val_losses=[],[]
    train_accs, val_accs= [], []

    for epoch in range(num_epochs):
        model.train()

        for input_batch, target_batch in train_loader:
            inputs, targets=input_batch.to(device), target_batch.to(device)
            optimizer.zero_grad()
            logits=model(inputs)

            loss= torch.nn.functional.cross_entropy(logits, targets)
            loss.backward()
            optimizer.step

        if epoch%eval_freq==0:
            train_loss= calc_loss_loader(train_loader, model, device, eval_iter)
            val_loss= calc_loss_loader(val_loader, model, device, eval_iter)

            train_acc= calc_accuracy_loader(train_loader, model, device, eval_iter)
            val_acc= calc_accuracy_loader(val_loader, model, device, eval_iter)

            train_losses.append(train_loss)
            val_losses.append(val_loss)
            train_accs.append(train_acc)
            val_accs.append(val_acc)

            print("f Epoch{epoch+1:02d} | train loss:{train_loss:.4f} | val loss{val_loss:.4f}")
            print(f"f | train acc : {train_acc*100:.2f}% | val acc:{val_acc*100:.2f}%")
    return train_losses, val_losses, train_accs, val_accs

## Summary

Pretraining taught the model language. This chapter taught it a job.

- **Fine-tuning** reuses a pretrained GPT instead of starting from random weights. The transformer already understands English; we only teach it how to turn that understanding into a label.
- **`GPTForClassification`** keeps the embeddings, the transformer stack, and the final LayerNorm, and replaces the 50,257-way next-token head with a tiny `Linear(emb_dim, num_classes)` layer.
- **Last-token pooling** (`x[:, -1, :]`) is how a causal model summarises a whole message. The final position has been allowed to attend to every token before it, so its vector is the right input for the classifier.
- **Freezing the backbone** locks about 124 million pretrained parameters and leaves only the new head trainable — 1,538 numbers for a 768-dimensional binary classifier. That is why a small labelled dataset is enough.
- A **classification dataset** is one label per text, not one next-token ID per position. Messages are encoded with the GPT-2 tokenizer and right-padded with the end-of-text token so they form a rectangular batch.
- **Accuracy** answers the human question, "how many labels did we get right?", and sits next to cross-entropy so we can tell confident mistakes from actual progress.
- The **training loop** is Chapter 4's recipe with a smaller output: logits of shape `(batch, num_classes)` and targets of shape `(batch,)`, with no flattening.

The model can now be specialised for any label set the same way: swap `num_classes`, point `SpamDataset` at a new CSV, and train the head. A later step in the series is the other flavour of fine-tuning — **instruction fine-tuning** — where the model learns to follow a prompt rather than pick from a fixed list of classes.